# Visualising TypeDB results in python
This tutorial builds a simple python library for visualising TypeDB results as graphs using 
the query representation returned by the analyze endpoint, 
and optionally returned with a query when the `include_query_structure` `QueryOption` is set.
We assume the user has is familiar with the TypeDB python driver ([tutorial](TODO:TUTORIAL_LINK))
It uses the python driver for interacting with TypeDB, networkx for building the graph, and matplotlib for visualisation.
We chose networkx and matplotlib purely because they're widely used, but you could use any - 
TypeDB studio uses sigmajs and graphology.

The full code of this tutorial is available on [github](TODO:GITHUB_LINK). It is a pyproject that can be built, published and installed.



## Running example:
For the following sections, we'll use this toy dataset.


In [ ]:
SCHEMA = """
define
  attribute name, value string;
  attribute age, value integer;
  entity person, owns name, owns age;
"""

DATA = """
insert
  $john isa person, has name "John", has age 20;
  $jane isa person, has name "Jane", has age 30; 
"""

SIMPLE_QUERY = """
    match $x isa person, has name $n;
"""


In [ ]:
from typedb.driver import TypeDB, Credentials, DriverOptions, TypeDB, QueryOptions, TransactionType

DB_ADDRESS = "127.0.0.1:1729"
DB_CREDENTIALS = Credentials("admin", "password")
DRIVER_OPTIONS = DriverOptions(is_tls_enabled=False)
QUERY_OPTIONS = QueryOptions()
QUERY_OPTIONS.include_query_structure = True
DB_NAME = "typedb-graph-tutorial-py"

def setup(driver, schema, data):
    if DB_NAME in [db.name for db in driver.databases.all()]:
        driver.databases.get(DB_NAME).delete()
    driver.databases.create(DB_NAME)
    with driver.transaction(DB_NAME, TransactionType.SCHEMA) as tx:
        tx.query(schema).resolve()
        tx.commit()
    with driver.transaction(DB_NAME, TransactionType.WRITE) as tx:
        rows = list(tx.query(data).resolve())
        assert 1 == len(rows)
        tx.commit()
        
driver = TypeDB.driver(DB_ADDRESS, DB_CREDENTIALS, DRIVER_OPTIONS)
setup(driver, SCHEMA, DATA)


## Background: Analyzing queries and PipelineStructure  
TypeDB 3.7 introduces the `analyze` operation. Analyzing a query returns its pipeline structure, along with typing information for the variables in each pattern.

A pipeline structure is a representation of a query pipeline. A pipeline is made up of `PipelineStage`s, some of which are made of `Conjunction`s.
```python
class Pipeline:
    def stages(self) -> Iterator[PipelineStage]
    def conjunction(self, conjunction_id: ConjunctionID) -> Optional[Conjunction]
    # ...

class MatchStage(PipelineStage):
    def block(self) -> "ConjunctionID"

class InsertStage(PipelineStage):
    def block(self) -> "ConjunctionID"    

```

In [ ]:
# Analyze the simple query
from typedb.analyze import Pipeline, Constraint, ConstraintVertex

with driver.transaction(DB_NAME, TransactionType.READ) as tx:
    analyzed = tx.analyze(SIMPLE_QUERY).resolve()
pipeline = analyzed.pipeline()
stages = list(pipeline.stages())
stages

A conjunction is a collection of `Constraint`s, which should be familiar from the TypeQL constructs:
```python
class Conjunction:
    def constraints(self) -> Iterator["Constraint"]


class Isa(Constraint, ABC):
    """Represents an 'isa' constraint: <instance> isa(!) <type>"""
    def instance(self) -> "ConstraintVertex"
    def type(self) -> "ConstraintVertex"
    def exactness(self) -> "ConstraintExactness" # isa or isa!


class Has(Constraint, ABC):
    """Represents a 'has' constraint: <owner> has <attribute>"""
    def owner(self) -> "ConstraintVertex"
    def attribute(self) -> "ConstraintVertex"
```
TypeQL constructs can be seen as constraints between one or more `ConstraintVertex`es - 
These can be type `Label`s, `Variable`s or raw `Value`s.

In [ ]:
# Get the constraints from the match stage
match_stage = stages[0]
conjunction = pipeline.conjunction(match_stage.block())
constraints = list(conjunction.constraints())
constraints

## From constraints to the query graph
Before we get to visualising the data returned by queries, We'll turn the constraints in our query into a graph and visualise it.
We'll use a `to_query_edge` function to go from a Constraint to a labelled edge represented by the tuple `(from, label, to)`

In [ ]:
# Convert all constraints to edges
from typing import Tuple
def to_query_edge(constraint: Constraint) -> Tuple[ConstraintVertex, str, ConstraintVertex]:
    if constraint.is_isa():
        isa = constraint.as_isa()
        return (isa.instance(), "isa", isa.type())
    elif constraint.is_has():
        has = constraint.as_has()
        return (has.owner(), "has", has.attribute())
    else:
        raise NotImplementedError("Not implemented in tutorial.")

query_edges = [to_query_edge(constraint) for constraint in constraints]
query_edges

In [ ]:
import networkx
from matplotlib import pyplot

query_graph = networkx.MultiDiGraph()
for (u, label, v) in query_edges:
    query_graph.add_edge(u,v)
node_labels = {node: str(node) for node in query_graph.nodes()}
networkx.draw(query_graph, labels=node_labels)
pyplot.show()

### Better formatting
That's the query graph, but it's of little use to the viewer. extra function and pass it into the drawing functions. We'll create a `node_style`  which specifies some style attributes, and a `draw` function which uses these to draw the networkx graph using matplotlib.

In [ ]:
from typing import Dict, List, Tuple
def node_style(pipeline: Pipeline, node: ConstraintVertex) -> Dict[str, any]:
    color = "g" if node.is_variable() else "b"
    label = ("$" + pipeline.get_variable_name(node.as_variable())) if node.is_variable() else str(node)
    shape = "s" if node.is_label() else "o"
    return {
        "color": color,
        "label": label,
        "shape": shape,
    }

def draw(edges: List[Tuple[ConstraintVertex, str, ConstraintVertex]], node_styles: Dict[ConstraintVertex, Dict[str, any]]):
    graph = networkx.MultiDiGraph()
    graph.add_edges_from((u,v, label) for (u, label, v) in edges)
    pos = networkx.forceatlas2_layout(graph) if hasattr(networkx, 'forceatlas2_layout') else networkx.spectral_layout(graph)
    
    nodes_by_shape = {node_styles[n]["shape"]: [] for n in graph.nodes}
    for node in nodes:
        nodes_by_shape[node_styles[node]["shape"]].append(node)    
    
    for (shape, node_subset) in nodes_by_shape.items():
        node_colors = [node_styles[n]["color"] for n in node_subset]
        node_labels = {n: node_styles[n]["label"] for n in node_subset}
        networkx.draw_networkx_nodes(graph, pos, nodelist=nodes_by_shape[shape], node_color=node_colors, node_shape=shape)
        networkx.draw_networkx_labels(graph, pos, labels = node_labels)
    networkx.draw_networkx_edges(graph, pos)
    
    edge_labels = { (u,v): label for (u, label, v) in edges}
    networkx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels)
    pyplot.show()

# Prepare node styles
nodes = set(u for (u,_,_) in query_edges).union(set(v for (_,_,v) in query_edges))
node_styles = { n: node_style(pipeline, n) for n in nodes}
# Draw
draw(query_edges, node_styles)


## Query results
TypeDB answers are `ConceptRow`s which map variables in the queries to concepts in the database. 
It has a similar interface to a python dictionary. If the `include_query_structure` `QueryOption` is set, the pipeline structure will be returned and each row will have shared access to it.
```python
class ConceptRow:
    def column_names(self) -> Iterator[str] # keys
    def concepts(self) -> Iterator[Concept]  # values
    def get(self, column_name: str) -> Optional[Concept] # get

    def query_structure(self) -> Optional["Pipeline"] # Shared access to the pipeline structure
```



In [ ]:
with driver.transaction(DB_NAME, TransactionType.READ) as tx:
    answers = list(tx.query(SIMPLE_QUERY, QUERY_OPTIONS).resolve())
assert 2 == len(answers), "TypeDB answer count mismatch"
answers

In [ ]:
# Every answer also has a reference to the pipeline structure
list(answers[0].query_structure().stages())

## From query structure and rows to graphs
The graph view sees pattern matching as finding a homomorphism between the query-graph and the database. 
An answer is thus the mapping between the two graphs.
To construct the graph representing the answer, we simply have to 
substitute the concepts in the answers for the variables in the associated constraints.


In [ ]:
from typedb.driver import ConceptRow, Concept
def substitute(pipeline: Pipeline, vertex: ConstraintVertex, row: ConceptRow) -> Concept:
    if vertex.is_label():
        return vertex.as_label()
    elif vertex.is_variable():
        var_name = pipeline.get_variable_name(vertex.as_variable())
        return row.get(var_name) if var_name else None
    else:
        raise NotImplementedError("Not implemented in tutorial. See resolve_constraint_vertex")

answers_as_data_edges = [
    [(substitute(pipeline, u, row), label, substitute(pipeline, v, row)) for (u,label,v) in query_edges]
    for row in answers
]
answers_as_data_edges

### Visualising
We'll need a new node style since our vertices are now concepts rather than `ConstraintVertex`es

In [ ]:
# We need to update our node_style
def data_node_style(node: Concept) -> Dict[str, any]:
    color = "g" if node.is_type() else "b"
    if node.is_type():
        label = str(node)
    elif node.is_attribute():
        label = f"{node.get_type().get_label()}:{node.get_value()}"
    else:
        label = f"{node.get_type().get_label()}#{node.get_iid()[-4:]}"
    
    shape = "s" if (node.is_attribute() or node.is_attribute_type()) else "o"
    return {
        "color": color,
        "label": label,
        "shape": shape,
    }

# Flatten them and remove any duplicate edges:
data_edges = set(e for answer_edges in answers_as_data_edges for e in answer_edges)

# Prepare node styles
nodes = set(u for (u,_,_) in data_edges).union(set(v for (_,_,v) in data_edges))
node_styles = { n: data_node_style(n) for n in nodes}
# Draw
draw(data_edges, node_styles)
